<a href="https://colab.research.google.com/github/willow788/AI-learns-to-Make-Art/blob/main/PDFSummarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -qqq --upgrade langchain langchain-community pypdf faiss-cpu transformers sentence-transformers langchain-huggingface

In [6]:
!pip install langchain_Classic

In [10]:
print("Re-evaluating LocalRAGSystem class...")
import os
import logging
from google.colab import files
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings # Corrected import
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_classic.chains import RetrievalQA # Reverted as per user's instruction

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class LocalRAGSystem:
  def __init__(self):
    self.documents = []
    self.vector_store = None
    self.embeddings = None
    self.llm = None
    self.qa_chain = None
    self.documents_chunks = []

  def upload_pdfs(self):
    uploaded = files.upload()
    pdf_paths = list(uploaded.keys())
    logger.info(f"Uploaded PDF: {pdf_paths}")
    return pdf_paths

  def load_documents(self, pdf_paths):
    for pdf_path in pdf_paths:
      loader = PyPDFLoader(pdf_path)
      documents = loader.load()
      self.documents.extend(documents)
    logger.info(f"Loaded {len(self.documents)} pages in total.")

  def split_documents(self, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    self.documents_chunks = text_splitter.split_documents(self.documents)
    logger.info(f"Splitted into {len(self.documents_chunks)} chunks.")

  def setup_embeddings(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    self.embeddings = HuggingFaceEmbeddings(model_name=model_name)
    logger.info(f"Embedding model {model_name} loaded.")

  def create_vector_store(self):
    self.vector_store = FAISS.from_documents(self.documents_chunks, self.embeddings)
    logger.info(f"Created the FAISS vector store.")

  def setup_local_llm(self, model_id="google/flan-t5-base", device="auto"):
      tokenizer = AutoTokenizer.from_pretrained(model_id)
      model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map=device)
      pipe = pipeline("text2text-generation", model=model,
                      tokenizer=tokenizer, max_new_tokens=512, temperature=0.7)
      self.llm = HuggingFacePipeline(pipeline=pipe)
      logger.info(f"Local LLM {model_id} ready.")

  def setup_qa_chain(self, k=3):
      self.qa_chain = RetrievalQA.from_chain_type(
          llm=self.llm,
          chain_type="stuff",
          retriever=self.vector_store.as_retriever(search_kwargs={"k": k})
      )
      logger.info(f"Retrieval QA chain set with top {k} documents retrieved.")

  def answer_question(self, question):
      answer = self.qa_chain.run(question)
      logger.info(f"Answered question: {question}")
      return answer

  def run_setup(self, chunk_size=1000, chunk_overlap=200, model_id="google/flan-t5-base", k=3):
      pdf_paths = self.upload_pdfs()
      self.load_documents(pdf_paths)
      self.split_documents(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
      self.setup_embeddings()
      self.create_vector_store()
      self.setup_local_llm(model_id=model_id)
      self.setup_qa_chain(k=k)
      logger.info("RAG summarizer is ready to answer questions.")

if __name__ == "__main__":
    rag = LocalRAGSystem()
    rag.run_setup()

    q1 = "What is the main topic of these documents?"
    print(f"Q: {q1}\nA: {rag.answer_question(q1)}")

    q2 = "Summarize the key points from the documents."
    print(f"Q: {q2}\nA: {rag.answer_question(q2)}")

Re-evaluating LocalRAGSystem class...


Saving short-stories-jack-and-the-beanstalk-transcript.pdf to short-stories-jack-and-the-beanstalk-transcript (2).pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"